# Sesión 20 — Aprendizaje Autosupervisado y Contrastivo
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo V · Arquitecturas Avanzadas y AI Generativa — Sesión de cierre**

## Objetivos de aprendizaje

1. Comprender el paradigma de aprendizaje autosupervisado y por qué importa cuando las etiquetas son escasas.
2. Derivar la **pérdida contrastiva InfoNCE / NT-Xent** y comprender el papel de los negativos.
3. Implementar **SimCLR** para preentrenamiento de señales fisiológicas.
4. Implementar **codificación enmascarada (MAE)** para aprendizaje de representaciones de EEG.
5. Evaluar representaciones con sondeo lineal (linear probing) y ajuste fino con pocos ejemplos.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Chen, T. et al. (2020). A simple framework for contrastive learning of visual representations (SimCLR). *ICML*. |
| ★★★ | He, K. et al. (2022). Masked autoencoders are scalable vision learners (MAE). *CVPR*. |
| ★★☆ | Oord, A. et al. (2018). Representation learning with contrastive predictive coding (CPC). *arXiv*. |
| ★★☆ | Mohsenvand, M.N. et al. (2020). Contrastive representation learning for electroencephalogram classification. *NeurIPS ML4H*. |
| ★★☆ | Cheng, J. et al. (2020). Subject-independent EEG-based emotion recognition using adversarial learning. *IEEE TAFFC*. |
| ★☆☆ | Resumen de SSL de Lilian Weng: https://lilianweng.github.io/posts/2021-05-31-contrastive/ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

rng    = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print(f'Dispositivo: {device}')

## Parte 1 — ¿Por qué aprendizaje autosupervisado para señales biomédicas?

El problema central en ML biomédico:

- **Datos sin etiquetar** son abundantes: grabaciones continuas de EEG, Holter ECG de 24h, flujos de wearables
- **Datos etiquetados** son costosos: requieren tiempo de clínicos, acuerdo entre evaluadores, aprobación ética
- **El aprendizaje autosupervisado** (SSL) preentrena en datos sin etiquetar, luego ajusta finamente con pocas etiquetas

La idea clave: **definir una tarea pretexto** cuya solución requiera aprender representaciones útiles — sin ninguna etiqueta humana.

In [ ]:
# ── Simular dataset de EEG: abundante sin etiquetar + escaso etiquetado ──────
# Tarea: clasificar alerta (0) vs somnoliento (1) a partir de características
# espectrales de EEG de 8 canales
# Escenario: 1000 sujetos sin etiquetar, solo 50 etiquetados

N_SIN_ETIQUETAR = 1000
N_ETIQUETADOS   = 200    # total; barreremos sobre subconjuntos más pequeños
N_FEATURES      = 8      # delta/theta/alfa/beta/gamma × 2 canales
SEG_LEN         = 64     # longitud del vector de características tras aplanar

def crear_muestra_eeg(label, rng_):
    """Simula un vector de características espectrales de EEG de 8 dimensiones."""
    if label == 0:  # alerta
        x = rng_.normal([5, 4, 20, 10, 3, 5, 4, 18], 2, N_FEATURES)
    else:           # somnoliento
        x = rng_.normal([8, 14, 12, 6, 2, 7, 13, 10], 2.5, N_FEATURES)
    return x.astype(np.float32)

# Conjunto sin etiquetar (las etiquetas se ocultan al modelo durante el preentrenamiento)
y_sin_etiquetar = rng.integers(0, 2, N_SIN_ETIQUETAR)
X_sin_etiquetar = np.array([crear_muestra_eeg(y, rng) for y in y_sin_etiquetar])

# Conjunto etiquetado
y_etiquetado = rng.integers(0, 2, N_ETIQUETADOS)
X_etiquetado = np.array([crear_muestra_eeg(y, rng) for y in y_etiquetado])

# Normalizar
scaler_ssl = StandardScaler().fit(X_sin_etiquetar)
X_unl_s = scaler_ssl.transform(X_sin_etiquetar).astype(np.float32)
X_lab_s = scaler_ssl.transform(X_etiquetado).astype(np.float32)

print(f'Conjunto sin etiquetar: {X_unl_s.shape}')
print(f'Conjunto etiquetado:    {X_lab_s.shape}  (balance de clases: {y_etiquetado.mean():.2f})')

## Parte 2 — SimCLR: aprendizaje contrastivo con pérdida NT-Xent

SimCLR crea dos **vistas aumentadas** de cada muestra, luego acerca las representaciones
de la misma muestra (par positivo) mientras aleja las de muestras distintas (negativos):

$$\ell(i,j) = -\log \frac{\exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_j)/\tau)}{\sum_{k=1}^{2N} \mathbf{1}_{[k\neq i]}\,\exp(\text{sim}(\mathbf{z}_i,\mathbf{z}_k)/\tau)}$$

donde $\tau$ es un parámetro de temperatura y $\text{sim}$ es similitud coseno.

In [ ]:
def aumentar_eeg(x, rng_t=None):
    """Aumentaciones para características espectrales de EEG."""
    rng_t = rng_t or torch.Generator(device=x.device)
    x = x.clone()
    # Jitter: añadir ruido gaussiano pequeño
    x = x + 0.05 * torch.randn_like(x)
    # Dropout de canal: poner a cero un canal aleatorio
    ch = torch.randint(0, x.size(-1), (1,)).item()
    x[..., ch] = 0.0
    # Escalado de amplitud
    x = x * (0.9 + 0.2 * torch.rand(1, device=x.device))
    return x


def perdida_nt_xent(z1, z2, temperature=0.5):
    """
    Pérdida NT-Xent (entropía cruzada normalizada con escalado de temperatura).
    z1, z2: (B, d) — dos vistas aumentadas del mismo batch.
    """
    B = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)   # (2B, d)

    # Matriz de similitud
    sim = torch.mm(z, z.T) / temperature   # (2B, 2B)

    # Enmascarar la autosimilitud
    mask = torch.eye(2*B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float('-inf'))

    # Pares positivos: (i, i+B) y (i+B, i)
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    loss   = F.cross_entropy(sim, labels)
    return loss


class SimCLR_Encoder(nn.Module):
    """Encoder f(·) + cabeza de proyección g(·) para preentrenamiento contrastivo."""
    def __init__(self, input_dim=8, hidden=64, proj_dim=32):
        super().__init__()
        # Encoder base (backbone)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),
        )
        # Cabeza de proyección (solo se usa en preentrenamiento, se descarta en fine-tuning)
        self.projector = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, proj_dim)
        )

    def forward(self, x):
        h = self.encoder(x)     # representación (se usa para tareas posteriores)
        z = self.projector(h)   # proyección (se usa para la pérdida contrastiva)
        return h, z


# Preentrenar en datos sin etiquetar
encoder_ssl = SimCLR_Encoder(input_dim=8, hidden=64, proj_dim=32).to(device)
opt_ssl     = optim.Adam(encoder_ssl.parameters(), lr=3e-4, weight_decay=1e-4)
loader_ssl  = DataLoader(TensorDataset(torch.tensor(X_unl_s)),
                          batch_size=128, shuffle=True, drop_last=True)

print('Preentrenando el encoder SimCLR (datos sin etiquetar)...')
hist_ssl = []
for ep in range(60):
    encoder_ssl.train()
    ep_loss = 0
    for (Xb,) in loader_ssl:
        Xb = Xb.to(device)
        # Dos vistas aumentadas
        x1 = aumentar_eeg(Xb)
        x2 = aumentar_eeg(Xb)
        _, z1 = encoder_ssl(x1)
        _, z2 = encoder_ssl(x2)
        loss   = perdida_nt_xent(z1, z2, temperature=0.5)
        opt_ssl.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(encoder_ssl.parameters(), 1.0)
        opt_ssl.step()
        ep_loss += loss.item()
    hist_ssl.append(ep_loss / len(loader_ssl))
    if (ep+1) % 15 == 0:
        print(f'  Época {ep+1:3d}  pérdida NT-Xent = {hist_ssl[-1]:.4f}')

print('Preentrenamiento completo.')

## Parte 3 — Sondeo lineal: evaluando la calidad de la representación

In [ ]:
def extraer_caracteristicas(encoder, X_np, device):
    """Extrae representaciones del encoder (backbone, sin el proyector)."""
    encoder.eval()
    with torch.no_grad():
        h, _ = encoder(torch.tensor(X_np).to(device))
    return h.cpu().numpy()


# También entrenamos un encoder aleatorio (línea base) y un MLP supervisado
encoder_random = SimCLR_Encoder(input_dim=8, hidden=64, proj_dim=32).to(device)  # sin entrenar

X_tr_lab, X_te_lab, y_tr_lab, y_te_lab = train_test_split(
    X_lab_s, y_etiquetado, test_size=0.25, stratify=y_etiquetado, random_state=0)

# Regímenes de pocas etiquetas a probar
n_label_sizes = [5, 10, 20, 50, 100, 150]

resultados_sonda = {'SimCLR (sonda lineal)': [], 'Inicialización aleatoria (sonda lineal)': [],
                     'Características crudas (sin SSL)': []}

for n_lab in n_label_sizes:
    # Usar las primeras n_lab muestras para entrenamiento
    Xtr_n = X_tr_lab[:n_lab]
    ytr_n = y_tr_lab[:n_lab]

    for enc, name in [(encoder_ssl,    'SimCLR (sonda lineal)'),
                       (encoder_random, 'Inicialización aleatoria (sonda lineal)')]:
        h_tr = extraer_caracteristicas(enc, Xtr_n,   device)
        h_te = extraer_caracteristicas(enc, X_te_lab, device)
        clf  = LogisticRegression(max_iter=500, C=1.0)
        clf.fit(h_tr, ytr_n)
        prob = clf.predict_proba(h_te)[:,1]
        resultados_sonda[name].append(roc_auc_score(y_te_lab, prob))

    # Características crudas (sin encoder)
    clf_raw = LogisticRegression(max_iter=500, C=1.0)
    clf_raw.fit(Xtr_n, ytr_n)
    prob_raw = clf_raw.predict_proba(X_te_lab)[:,1]
    resultados_sonda['Características crudas (sin SSL)'].append(roc_auc_score(y_te_lab, prob_raw))


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores = ['steelblue', 'tomato', 'seagreen']
for (name, aurocs), color in zip(resultados_sonda.items(), colores):
    axes[0].plot(n_label_sizes, aurocs, '-o', lw=2.5, ms=7, color=color, label=name)
axes[0].axhline(0.5, color='gray', ls=':', lw=1)
axes[0].set(xlabel='Número de ejemplos de entrenamiento etiquetados',
            ylabel='AUROC (sonda lineal)', xscale='log',
            title='Eficiencia de datos: SimCLR vs líneas base\n'
                  'Sonda lineal = congelar encoder, entrenar solo el clasificador final')
axes[0].legend(fontsize=8)

# Visualización del espacio latente
h_all_ssl    = extraer_caracteristicas(encoder_ssl,    X_lab_s, device)
h_all_random = extraer_caracteristicas(encoder_random, X_lab_s, device)

pca = PCA(n_components=2)
for h_2d, titulo, ax in [
    (pca.fit_transform(h_all_ssl),    'Representaciones de SimCLR (PCA)', axes[1]),
]:
    for cls, label, color in [(0,'Alerta','steelblue'), (1,'Somnoliento','tomato')]:
        mask = y_etiquetado == cls
        ax.scatter(h_2d[mask,0], h_2d[mask,1],
                    alpha=0.5, s=18, color=color, label=label)
    ax.set(xlabel='PC1', ylabel='PC2', title=titulo)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\nComparación de AUROC con el conjunto etiquetado completo:')
for name, aurocs in resultados_sonda.items():
    print(f'  {name:42s}  {aurocs[-1]:.3f}')

## Parte 4 — Autoencoder Enmascarado (MAE) para EEG

MAE preentrena enmascarando una gran fracción de la entrada y aprendiendo a
reconstruirla. A diferencia de los métodos contrastivos, no requiere pares negativos
ni diseño de aumentación — solo enmascaramiento aleatorio.

Aplicado a espectrogramas de EEG: enmascarar 75% de los parches → el encoder ve solo
el 25% → el decoder reconstruye el resto.

In [ ]:
class MAE_EEG(nn.Module):
    """
    Autoencoder Enmascarado para secuencias de características espectrales de EEG.
    Trata las 8 características espectrales como 8 'parches', enmascara una fracción,
    el encoder ve solo los parches visibles, el decoder reconstruye todos.
    """
    def __init__(self, n_features=8, d_enc=64, d_dec=32, mask_ratio=0.75):
        super().__init__()
        self.mask_ratio   = mask_ratio
        self.n_features   = n_features

        # Encoder: procesa solo los tokens visibles (sin enmascarar)
        self.enc_proj = nn.Linear(1, d_enc)   # cada característica es un 'parche' 1-d
        self.enc_pos  = nn.Embedding(n_features, d_enc)
        enc_layer     = nn.TransformerEncoderLayer(
            d_model=d_enc, nhead=4, dim_feedforward=d_enc*4,
            batch_first=True, norm_first=True, dropout=0.1)
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=2)

        # Decoder: reconstruye todos los tokens desde visibles codificados + tokens de máscara
        self.dec_proj   = nn.Linear(d_enc, d_dec)
        self.mask_token = nn.Parameter(torch.randn(1, 1, d_dec))
        self.dec_pos    = nn.Embedding(n_features, d_dec)
        dec_layer       = nn.TransformerEncoderLayer(
            d_model=d_dec, nhead=4, dim_feedforward=d_dec*4,
            batch_first=True, norm_first=True, dropout=0.1)
        self.decoder    = nn.TransformerEncoder(dec_layer, num_layers=2)
        self.pred_head  = nn.Linear(d_dec, 1)   # predecir el escalar de cada característica

    def forward(self, x):
        """
        x: (B, n_features) — un vector de características espectrales de EEG
        Retorna: x_rec (B, n_features), mask (B, n_features) bool
        """
        B, F = x.shape
        n_mask = int(F * self.mask_ratio)

        # Máscara aleatoria por muestra
        noise = torch.rand(B, F, device=x.device)
        ids_sorted = noise.argsort(dim=1)
        ids_restore = ids_sorted.argsort(dim=1)
        mask_bool   = ids_sorted < n_mask   # True = enmascarado

        # ── Encoder (solo parches visibles) ────────────────────────────────
        x_tok = self.enc_proj(x.unsqueeze(-1))   # (B, F, d_enc)
        pos_idx = torch.arange(F, device=x.device).unsqueeze(0).expand(B, -1)
        x_tok   = x_tok + self.enc_pos(pos_idx)

        # Mantener solo los tokens visibles
        visible_mask = ~mask_bool   # (B, F)
        x_vis = x_tok[visible_mask].reshape(B, F - n_mask, -1)  # (B, n_vis, d_enc)
        h_vis = self.encoder(x_vis)                              # (B, n_vis, d_enc)

        # ── Decoder (todas las posiciones) ────────────────────────────────────────
        h_vis_d = self.dec_proj(h_vis)                    # (B, n_vis, d_dec)
        mask_tokens = self.mask_token.expand(B, n_mask, -1)   # (B, n_mask, d_dec)

        # Intercalar tokens visibles + de máscara en el orden original
        full_tokens = torch.zeros(B, F, self.mask_token.size(-1), device=x.device)
        full_tokens[visible_mask]  = h_vis_d.reshape(-1, h_vis_d.size(-1))
        full_tokens[mask_bool]     = mask_tokens.reshape(-1, mask_tokens.size(-1))

        dec_pos_emb = self.dec_pos(pos_idx)
        full_tokens = full_tokens + dec_pos_emb
        h_full  = self.decoder(full_tokens)             # (B, F, d_dec)
        x_rec   = self.pred_head(h_full).squeeze(-1)   # (B, F)

        return x_rec, mask_bool

    def loss(self, x, x_rec, mask):
        """Calcula la pérdida de reconstrucción solo en las posiciones enmascaradas."""
        return F.mse_loss(x_rec[mask], x[mask])


mae_model  = MAE_EEG(n_features=8, mask_ratio=0.75).to(device)
opt_mae    = optim.AdamW(mae_model.parameters(), lr=1e-3, weight_decay=1e-2)
loader_mae = DataLoader(TensorDataset(torch.tensor(X_unl_s)),
                         batch_size=128, shuffle=True, drop_last=True)

print('Preentrenando el encoder MAE (datos sin etiquetar)...')
hist_mae = []
for ep in range(60):
    mae_model.train()
    ep_loss = 0
    for (Xb,) in loader_mae:
        Xb = Xb.to(device)
        x_rec, mask = mae_model(Xb)
        loss = mae_model.loss(Xb, x_rec, mask)
        opt_mae.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(mae_model.parameters(), 1.0)
        opt_mae.step()
        ep_loss += loss.item()
    hist_mae.append(ep_loss / len(loader_mae))
    if (ep+1) % 15 == 0:
        print(f'  Época {ep+1:3d}  pérdida MAE = {hist_mae[-1]:.5f}')

print('Completo.')

In [ ]:
# ── Comparar todos los métodos SSL: SimCLR vs MAE vs Aleatorio vs Supervisado ─

def caracteristicas_mae(model, X_np, device):
    """Extrae representaciones del encoder MAE (salida del encoder con pooling de media)."""
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(X_np).to(device)
        # Ejecutar el encoder con todo visible (truco mask_ratio=0: forward completo, tomar todos los tokens)
        x_tok = model.enc_proj(Xt.unsqueeze(-1))
        pos   = torch.arange(model.n_features, device=device).unsqueeze(0).expand(Xt.size(0),-1)
        x_tok = x_tok + model.enc_pos(pos)
        h     = model.encoder(x_tok)  # (B, F, d_enc)
        return h.mean(1).cpu().numpy()  # pooling de media sobre características


resultados_todos = {}
n_label_sizes2 = [5, 10, 20, 50, 100, 150]

def _sonda_lineal(enc, Xtr, ytr, Xte, dev, feat_fn):
    h_tr = feat_fn(enc, Xtr, dev)
    h_te = feat_fn(enc, X_te_lab, dev)
    clf  = LogisticRegression(max_iter=500, C=1.0)
    clf.fit(h_tr, ytr)
    return roc_auc_score(y_te_lab, clf.predict_proba(h_te)[:,1])

def _sonda_cruda(Xtr, ytr, Xte=None):
    clf = LogisticRegression(max_iter=500, C=1.0)
    clf.fit(Xtr, ytr)
    return roc_auc_score(y_te_lab, clf.predict_proba(X_te_lab)[:,1])

configuraciones_metodo = [
    ('SimCLR',                  lambda Xtr, ytr, Xte: _sonda_lineal(encoder_ssl,    Xtr, ytr, Xte, device, extraer_caracteristicas)),
    ('MAE',                     lambda Xtr, ytr, Xte: _sonda_lineal(mae_model,      Xtr, ytr, Xte, device, caracteristicas_mae)),
    ('Encoder aleatorio',       lambda Xtr, ytr, Xte: _sonda_lineal(encoder_random, Xtr, ytr, Xte, device, extraer_caracteristicas)),
    ('Crudo (sin SSL)',         lambda Xtr, ytr, Xte: _sonda_cruda(Xtr, ytr, Xte)),
]

for nombre_metodo, sonda_fn in configuraciones_metodo:
    aurocs = []
    for n_lab in n_label_sizes2:
        Xtr_n = X_tr_lab[:n_lab]
        ytr_n = y_tr_lab[:n_lab]
        aurocs.append(sonda_fn(Xtr_n, ytr_n, X_te_lab))
    resultados_todos[nombre_metodo] = aurocs
    print(f'{nombre_metodo:22s}: AUROC @ 5={aurocs[0]:.3f}  @ 50={aurocs[3]:.3f}  @ 150={aurocs[-1]:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
colores_metodo = ['steelblue', 'darkorange', 'tomato', 'gray']
ls_metodo      = ['-', '-', '--', ':']
for (name, aurocs), color, ls in zip(resultados_todos.items(), colores_metodo, ls_metodo):
    ax.plot(n_label_sizes2, aurocs, ls+'o', lw=2.5, ms=7, color=color, label=name)

ax.axhline(0.5, color='lightgray', ls=':', lw=1)
ax.set(xlabel='Número de ejemplos etiquetados', ylabel='AUROC', xscale='log',
       title='Preentrenamiento autosupervisado: SimCLR vs MAE vs líneas base\n'
             'El SSL brilla en el régimen de pocas etiquetas (lado izquierdo)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Parte 5 — Síntesis del curso: el pipeline completo de ML

Esta sesión cierra el Módulo V y conecta con el Módulo VI. La figura a continuación
mapea cada método importante del curso a su posición en el panorama de eficiencia de
datos — la dimensión práctica clave para ingenieros biomédicos.

In [ ]:
# Mapa del curso: métodos vs régimen de datos
mapa_metodos = {
    # (n_etiquetados_aprox, rendimiento_aprox, módulo, marcador)
    'Regresión logística':       (50,   0.72, 'II',  'o'),
    'SVM':                       (40,   0.75, 'II',  's'),
    'Random forest':             (100,  0.78, 'II',  'D'),
    'XGBoost':                   (150,  0.81, 'II',  'P'),
    'MLP (Módulo IV)':           (200,  0.80, 'IV',  'o'),
    'CNN 1-D':                   (300,  0.85, 'IV',  's'),
    'ResNet':                    (400,  0.87, 'IV',  'D'),
    'BiLSTM':                    (300,  0.84, 'IV',  '^'),
    'Transformer':               (500,  0.88, 'V',   'o'),
    'Transferencia (congelado)': (50,   0.83, 'V',   '*'),
    'Transferencia (ajustado)':  (200,  0.90, 'V',   '*'),
    'SimCLR (5 etiquetas)':      (5,    0.71, 'V',   'P'),
    'SimCLR (50 etiquetas)':     (50,   0.84, 'V',   'P'),
    'MAE (50 etiquetas)':        (50,   0.82, 'V',   'h'),
}

colores_modulo = {'I': 'gray', 'II': 'steelblue', 'III': 'seagreen',
                  'IV': 'darkorange', 'V': 'tomato'}

fig, ax = plt.subplots(figsize=(12, 6))
for name, (n_lab, perf, mod, mk) in mapa_metodos.items():
    ax.scatter(n_lab, perf, s=120, marker=mk,
                color=colores_modulo[mod], zorder=5,
                edgecolors='white', linewidth=0.7)
    ax.annotate(name, (n_lab, perf), fontsize=7.5,
                 xytext=(6, 3), textcoords='offset points',
                 color=colores_modulo[mod])

# Sombreado de regiones
ax.axvspan(1,   50,  alpha=0.04, color='tomato',    label='Régimen de pocos datos (brilla el SSL)')
ax.axvspan(50,  300, alpha=0.04, color='steelblue', label='Régimen de datos medio')
ax.axvspan(300, 600, alpha=0.04, color='seagreen',  label='Régimen de muchos datos (brillan las redes profundas)')

from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                           markersize=9, label=f'Módulo {m}')
                    for m, c in colores_modulo.items() if m != 'I']
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

ax.set(xlabel='Ejemplos de entrenamiento etiquetados aproximados (escala log)',
       ylabel='AUROC típico en tarea biomédica',
       xscale='log', xlim=(3, 700), ylim=(0.65, 0.96),
       title='Mapa del curso: métodos vs disponibilidad de datos\n'
             'No hay un único ganador — el método correcto depende de tu régimen de datos')
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Ablación de aumentaciones.** Para SimCLR, elimina sistemáticamente cada aumentación
   (jitter, dropout de canal, escalado de amplitud) una a la vez. Reporta el AUROC tras
   sondeo lineal con 20 etiquetas. ¿Qué aumentación contribuye más? Explica por qué desde
   una perspectiva de procesamiento de señales.

2. **Sensibilidad a la temperatura.** Entrena SimCLR con temperaturas τ ∈ {0.1, 0.3, 0.5,
   1.0, 2.0}. Para cada una, grafica la curva de pérdida contrastiva y el AUROC de sonda
   lineal con 20 etiquetas. ¿Qué temperatura funciona mejor y por qué? (Pista: considera
   qué le ocurre a la distribución softmax conforme τ → 0)

3. **BYOL (sin negativos).** Bootstrap Your Own Latent (Grill et al. 2020) logra
   aprendizaje contrastivo sin pares negativos usando una arquitectura de red
   online/objetivo. Implementa la regla de actualización de BYOL:
   $\mathcal{L} = 2 - 2\cdot\frac{q_\theta(z_\text{online})^\top\, z_\text{target}}{\|q_\theta(z_\text{online})\|\|z_\text{target}\|}$.
   Compara con SimCLR en el régimen de 20 etiquetas.

4. **SSL en EEG real.** Descarga el dataset PhysioNet EEG Motor Movement (109 sujetos,
   tareas MI + ME). Preentrena SimCLR en los primeros 80 sujetos (sin etiquetar), luego
   ajusta finamente en los últimos 29 sujetos con presupuestos de etiquetas variables.
   Reporta el AUROC LOSO vs la línea base supervisada. Grafica la curva de pérdida de
   preentrenamiento.

5. *(Desafío)* **SSL adaptativo de dominio.** Preentrena SimCLR en datos de EEG fuente
   (ej. BCI Competition IV 2a, 9 sujetos) y prueba el sondeo lineal en un dataset objetivo
   distinto (ej. PhysioNet Motor Movement). Compara la transferencia entre datasets con
   entrenar desde cero en el dataset objetivo. Añade una pérdida adversarial de dominio
   (estilo DANN) durante el preentrenamiento y mide la mejora.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| PhysioNet EEG Motor Movement | https://physionet.org/content/eegmmidb/ | 109 sujetos, MI + ME |
| BCI Competition IV 2a | https://www.bbci.de/competition/iv/ | MI de 4 clases, 9 sujetos |
| DREAMER (EEG/ECG) | https://zenodo.org/record/546113 | Reconocimiento de emociones |